# Add ending symbol to the text column

This notebook reads an Excel file, adds `।` to the end of values in the `text` column only, and saves the updated file as a new Excel file.

In [17]:
from pathlib import Path

import pandas as pd

INPUT_FILE = Path("wav_text.xlsx")
INPUT_SHEET = "normalized_text"
OUTPUT_FILE = Path("wav_text_with_ending_symbol.xlsx")
TEXT_COLUMN = "Text"
ENDING_SYMBOL = "।"

if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input workbook not found: {INPUT_FILE.resolve()}")

df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)

if TEXT_COLUMN not in df.columns:
    raise KeyError(f"Column '{TEXT_COLUMN}' not found. Available columns: {list(df.columns)}")

def add_ending_symbol(value):
    if isinstance(value, str):
        text = value.rstrip()
        if text and not text.endswith(ENDING_SYMBOL):
            return text + ENDING_SYMBOL
    return value

df[TEXT_COLUMN] = df[TEXT_COLUMN].map(add_ending_symbol)
df.to_excel(OUTPUT_FILE, index=False, sheet_name=INPUT_SHEET)

print(f"Saved updated file to: {OUTPUT_FILE.resolve()}")
print(f"Updated column: {TEXT_COLUMN}")

Saved updated file to: C:\Users\rakib\OneDrive\Desktop\dev\wav_text_with_ending_symbol.xlsx
Updated column: Text


---

---

# WAV Text Review Notebook
 
 Run the setup cell first. Then run the review interface cell to review every remaining WAV file from `wavs/`.
 
 Progress is saved after every item to `wav_text_review_progress.xlsx`, so you can stop with `q` and resume later.
 
 Controls: type `n` to save current text and go next, `p` to go back within this session, `q` to stop, or type replacement text to save it and go next.

In [18]:
from pathlib import Path
from datetime import datetime
import html
import re

import pandas as pd
from IPython.display import Audio, Markdown, display

try:
    import ipywidgets as widgets
except ImportError as exc:
    widgets = None
    WIDGETS_IMPORT_ERROR = exc
else:
    WIDGETS_IMPORT_ERROR = None

def detect_project_dir():
    required_names = ("wav_text_with_ending_symbol.xlsx", "wavs")
    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if all((root / name).exists() for name in required_names):
            return root.resolve()
    return Path.cwd().resolve()


PROJECT_DIR = detect_project_dir()
WAVS_DIR = (PROJECT_DIR / "wavs").resolve()
SOURCE_XLSX = (PROJECT_DIR / "wav_text_with_ending_symbol.xlsx").resolve()
OUTPUT_XLSX = (PROJECT_DIR / "wav_text_review_progress.xlsx").resolve()
SOURCE_SHEET = "normalized_text"
OUTPUT_SHEET = "reviewed_text"

if not SOURCE_XLSX.exists():
    raise FileNotFoundError(f"Source workbook not found: {SOURCE_XLSX.resolve()}")

if SOURCE_XLSX.resolve() == OUTPUT_XLSX.resolve():
    raise ValueError("SOURCE_XLSX and OUTPUT_XLSX must be different files.")


def natural_key(path_or_name):
    name = Path(path_or_name).name.lower()
    return [int(part) if part.isdigit() else part for part in re.split(r"(\d+)", name)]


def load_source_dataframe():
    df = pd.read_excel(SOURCE_XLSX, sheet_name=SOURCE_SHEET)
    expected = {"Audio Id", "Text"}
    missing = expected.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required column(s) in source workbook: {sorted(missing)}")

    df = df[["Audio Id", "Text"]].copy()
    df["Audio Id"] = df["Audio Id"].astype(str).str.strip()
    df["Text"] = df["Text"].fillna("").astype(str)
    return df


def load_progress_dataframe():
    columns = ["Audio Id", "Original Text", "Text", "Decision", "Updated At"]
    if not OUTPUT_XLSX.exists():
        return pd.DataFrame(columns=columns)

    workbook = pd.ExcelFile(OUTPUT_XLSX)
    sheet_name = OUTPUT_SHEET if OUTPUT_SHEET in workbook.sheet_names else workbook.sheet_names[0]
    df = pd.read_excel(OUTPUT_XLSX, sheet_name=sheet_name)
    for col in columns:
        if col not in df.columns:
            df[col] = ""

    df = df[columns].copy()
    df["Audio Id"] = df["Audio Id"].astype(str).str.strip()
    df["Original Text"] = df["Original Text"].fillna("").astype(str)
    df["Text"] = df["Text"].fillna("").astype(str)
    df["Decision"] = df["Decision"].fillna("").astype(str)
    df["Updated At"] = df["Updated At"].fillna("").astype(str)
    df = df.drop_duplicates(subset=["Audio Id"], keep="last")
    return df


def save_progress(df):
    ordered = df.sort_values("Audio Id", key=lambda series: series.map(natural_key)).reset_index(drop=True)
    ordered.to_excel(OUTPUT_XLSX, index=False, sheet_name=OUTPUT_SHEET)


def get_wav_files():
    if not WAVS_DIR.exists():
        raise FileNotFoundError(f"WAV folder not found: {WAVS_DIR.resolve()}")

    files = sorted(WAVS_DIR.glob("*.wav"), key=natural_key)
    if not files:
        raise FileNotFoundError(f"No .wav files found in: {WAVS_DIR.resolve()}")
    return files


def load_review_state():
    source_df = load_source_dataframe()
    progress_df = load_progress_dataframe()
    source_map = dict(zip(source_df["Audio Id"], source_df["Text"]))
    wav_files = get_wav_files()
    reviewed_ids = set(progress_df["Audio Id"]) if not progress_df.empty else set()
    pending_wavs = [wav_path for wav_path in wav_files if wav_path.name not in reviewed_ids]
    missing_wavs = [audio_id for audio_id in source_map if not (WAVS_DIR / audio_id).exists()]
    extra_wavs = [wav_path.name for wav_path in wav_files if wav_path.name not in source_map]
    return {
        "source_df": source_df,
        "progress_df": progress_df,
        "source_map": source_map,
        "wav_files": wav_files,
        "reviewed_ids": reviewed_ids,
        "pending_wavs": pending_wavs,
        "missing_wavs": missing_wavs,
        "extra_wavs": extra_wavs,
    }


state = load_review_state()
display(Markdown(f"""
- Source workbook: `{SOURCE_XLSX}`
- Progress workbook: `{OUTPUT_XLSX}`
- WAV files found: `{len(state['wav_files'])}`
- Already reviewed: `{len(state['progress_df'])}`
- Remaining: `{len(state['pending_wavs'])}`
- Missing WAVs referenced by source workbook: `{len(state['missing_wavs'])}`
- WAVs without source text: `{len(state['extra_wavs'])}`
"""))


- Source workbook: `C:\Users\rakib\OneDrive\Desktop\dev\wav_text_with_ending_symbol.xlsx`
- Progress workbook: `C:\Users\rakib\OneDrive\Desktop\dev\wav_text_review_progress.xlsx`
- WAV files found: `1497`
- Already reviewed: `1316`
- Remaining: `181`
- Missing WAVs referenced by source workbook: `0`
- WAVs without source text: `31`


In [19]:
state = load_review_state()
pending_wavs = state["pending_wavs"]
progress_df = state["progress_df"]

if progress_df.empty:
    last_checkpoint = "Warning: no saved review data found in `wav_text_review_progress.xlsx`. Review will start from the first wav file."
else:
    last_row = progress_df.sort_values("Updated At").iloc[-1]
    last_checkpoint = f"Last checkpoint: `{last_row['Audio Id']}` at `{last_row['Updated At']}`. You can navigate back to saved items from the review file."

display(Markdown(last_checkpoint))
display(Markdown(f"Navigator items: `{len(state['wav_files'])}` | Pending items: `{len(pending_wavs)}` | Saved items: `{len(progress_df)}`"))

if state["missing_wavs"]:
    missing_preview = "\n".join(f"- `{audio_id}`" for audio_id in state["missing_wavs"][:10])
    display(Markdown("Source rows with no matching WAV file:\n" + missing_preview))

if state["extra_wavs"]:
    extra_preview = "\n".join(f"- `{audio_id}`" for audio_id in state["extra_wavs"][:10])
    display(Markdown("WAV files with no matching source text:\n" + extra_preview))

if pending_wavs:
    preview_names = "\n".join(f"- `{wav_path.name}`" for wav_path in pending_wavs[:10])
    display(Markdown("First files waiting for review:\n" + preview_names))
else:
    display(Markdown("All wav files have already been reviewed."))

Last checkpoint: `racordsFinal_2ndSession_1316.wav` at `2026-06-18T16:30:38`. You can navigate back to saved items from the review file.

Navigator items: `1497` | Pending items: `181` | Saved items: `1316`

WAV files with no matching source text:
- `racordsFinal_2ndSession_44.wav`
- `racordsFinal_2ndSession_174.wav`
- `racordsFinal_2ndSession_238.wav`
- `racordsFinal_2ndSession_239.wav`
- `racordsFinal_2ndSession_242.wav`
- `racordsFinal_2ndSession_244.wav`
- `racordsFinal_2ndSession_255.wav`
- `racordsFinal_2ndSession_256.wav`
- `racordsFinal_2ndSession_259.wav`
- `racordsFinal_2ndSession_283.wav`

First files waiting for review:
- `racordsFinal_2ndSession_1317.wav`
- `racordsFinal_2ndSession_1318.wav`
- `racordsFinal_2ndSession_1319.wav`
- `racordsFinal_2ndSession_1320.wav`
- `racordsFinal_2ndSession_1321.wav`
- `racordsFinal_2ndSession_1322.wav`
- `racordsFinal_2ndSession_1323.wav`
- `racordsFinal_2ndSession_1324.wav`
- `racordsFinal_2ndSession_1325.wav`
- `racordsFinal_2ndSession_1326.wav`

In [ ]:
if widgets is None:
    raise ImportError(
        "ipywidgets is required for the GUI review interface. Install/enable ipywidgets in this VS Code notebook environment."
    ) from WIDGETS_IMPORT_ERROR


class WavTextReviewApp:
    def __init__(self):
        self.state = load_review_state()
        self.progress_df = self.state["progress_df"].copy()
        self.source_map = self.state["source_map"]
        self.wav_files = self.state["wav_files"]
        self.session_wavs = list(self.wav_files)
        self.index = self.initial_index()
        self.stopped = False
        self.missing_in_source = []
        self.message = ""
        self.draft_texts = {}

        self.theme = widgets.HTML(
            value="""
<style>
.wav-review-shell {
    background: linear-gradient(180deg, #f7f4ee 0%, #efe8dc 100%);
    color: #1f2937;
    border: 1px solid #d6cfc2;
    border-radius: 14px;
    padding: 12px;
    box-shadow: 0 10px 26px rgba(84, 74, 57, 0.12);
}
.wav-review-card {
    background: rgba(255, 252, 247, 0.96);
    border: 1px solid #ddd3c2;
    border-radius: 10px;
    padding: 10px 12px;
    margin-bottom: 8px;
}
.wav-review-card h2,
.wav-review-card h3,
.wav-review-card p,
.wav-review-card div,
.wav-review-card small {
    color: #1f2937;
    margin: 0;
}
.wav-review-muted {
    color: #6b7280;
}
.wav-review-card p,
.wav-review-card div {
    line-height: 1.7;
    font-size: 14px;
}
.wav-review-card h3 {
    font-size: 15px;
    font-weight: 700;
}
.wav-review-audio {
    width: 100%;
    margin-top: 6px;
}
.wav-review-focus {
    background: #fffdf8;
    border: 1px solid #d8cdb9;
    border-radius: 10px;
    padding: 8px 10px;
}
</style>
"""
        )
        self.header = widgets.HTML()
        self.audio_output = widgets.Output()
        self.original_text = widgets.HTML()
        self.saved_text = widgets.HTML()
        self.command = widgets.Textarea(
            placeholder="Edit transcript here...",
            layout=widgets.Layout(width="100%", height="120px"),
        )
        self.command.style.text_color = "#1f2937"
        self.command.style.font_family = "Segoe UI, Noto Sans Bengali, sans-serif"
        self.command.layout.border = "1px solid #cdbfa9"
        self.command.layout.padding = "8px"
        self.next_button = widgets.Button(description="Next", button_style="success", tooltip="Save current text and go next")
        self.prev_button = widgets.Button(description="Previous", tooltip="Save current text and go back")
        self.quit_button = widgets.Button(description="Stop", button_style="warning", tooltip="Stop now and keep saved progress")
        self.status = widgets.HTML()
        self.editor_box = widgets.VBox([self.command])
        self.button_row = widgets.HBox([self.prev_button, self.next_button, self.quit_button])
        self.container = widgets.VBox([
            self.theme,
            self.header,
            self.audio_output,
            self.original_text,
            self.saved_text,
            self.editor_box,
            self.button_row,
            self.status,
        ])
        self.container.layout = widgets.Layout(border="1px solid #d6cfc2", padding="12px", width="100%")
        self.button_row.layout = widgets.Layout(justify_content="space-between", gap="8px")
        self.next_button.layout = widgets.Layout(width="96px")
        self.prev_button.layout = widgets.Layout(width="96px")
        self.quit_button.layout = widgets.Layout(width="96px")

        self.next_button.on_click(lambda _: self.save_and_next())
        self.prev_button.on_click(lambda _: self.previous_item())
        self.quit_button.on_click(lambda _: self.stop())

    def refresh_progress_df(self):
        self.progress_df = load_progress_dataframe()

    def initial_index(self):
        if not self.session_wavs:
            return 0
        if self.progress_df.empty:
            self.message = "<b>Warning:</b> No saved review data found. Starting from the first item."
            return 0
        reviewed_ids = set(self.progress_df["Audio Id"])
        for idx, wav_path in enumerate(self.session_wavs):
            if wav_path.name not in reviewed_ids:
                self.message = "<b>Resume:</b> Opened the first pending item. Use Previous to move through saved items."
                return idx
        last_row = self.progress_df.sort_values("Updated At").iloc[-1]
        last_audio_id = str(last_row["Audio Id"]).strip()
        for idx, wav_path in enumerate(self.session_wavs):
            if wav_path.name == last_audio_id:
                self.message = "<b>Resume:</b> All items are already saved. Showing the last saved item."
                return idx
        self.message = "<b>Warning:</b> Saved review data exists but the checkpoint file could not be matched to the WAV list."
        return 0

    def progress_lookup(self, audio_id, refresh=True):
        if refresh:
            self.refresh_progress_df()
        if self.progress_df.empty:
            return None
        matches = self.progress_df[self.progress_df["Audio Id"] == audio_id]
        if matches.empty:
            return None
        return matches.iloc[-1]

    def current_text_for(self, audio_id):
        if audio_id in self.draft_texts:
            return self.draft_texts[audio_id]
        saved = self.progress_lookup(audio_id)
        if saved is not None:
            return str(saved["Text"])
        return self.source_map.get(audio_id, "")

    def is_reviewed(self, audio_id):
        return self.progress_lookup(audio_id) is not None

    def resolve_wav_path(self, wav_path):
        candidate = Path(wav_path).expanduser()
        if not candidate.is_absolute():
            candidate = (PROJECT_DIR / candidate).resolve()
        if candidate.exists():
            return candidate
        fallback = (WAVS_DIR / candidate.name).resolve()
        if fallback.exists():
            return fallback
        return fallback

    def card_html(self, title, body, note=""):
        note_html = f'<p class="wav-review-muted" style="margin-top:8px;">{note}</p>' if note else ""
        return (
            '<div class="wav-review-card">'
            f'<h3 style="margin-bottom:6px;">{title}</h3>'
            f'<div style="white-space: pre-wrap;">{body}</div>'
            f'{note_html}'
            '</div>'
        )

    def render_audio(self, wav_path):
        wav_path = self.resolve_wav_path(wav_path)
        self.audio_output.clear_output(wait=True)
        with self.audio_output:
            if not wav_path.exists():
                display(Markdown(f"**Audio file not found:** `{wav_path}`"))
                return
            display(Audio(filename=str(wav_path), autoplay=True))

    def status_html(self, message):
        return f'<div class="wav-review-card"><div>{message}</div></div>' if message else ''

    def render(self):
        if not self.session_wavs:
            refreshed_state = load_review_state()
            self.header.value = self.card_html("Done", "No WAV files are available for review.")
            self.audio_output.clear_output()
            self.original_text.value = ""
            self.saved_text.value = ""
            self.command.disabled = True
            self.next_button.disabled = True
            self.prev_button.disabled = True
            self.quit_button.disabled = True
            self.status.value = self.status_html(f"<b>Total reviewed:</b> {len(refreshed_state['progress_df'])}")
            return

        if self.index >= len(self.session_wavs):
            refreshed_state = load_review_state()
            self.index = len(self.session_wavs) - 1
            self.message = "<b>Reached the end.</b> Showing the last item so you can still review saved entries."

        if self.index < 0:
            self.index = 0
            self.message = "<b>Already at the first item.</b>"

        wav_path = self.resolve_wav_path(self.session_wavs[self.index])
        audio_id = wav_path.name
        global_index = self.wav_files.index(wav_path) + 1
        original = self.source_map.get(audio_id, "")
        current = self.current_text_for(audio_id)
        saved_row = self.progress_lookup(audio_id)
        reviewed_label = "Saved in progress file" if saved_row is not None else "Not saved yet"
        updated_at = ""
        if saved_row is not None and str(saved_row["Updated At"]).strip():
            updated_at = f" | Last saved: {html.escape(str(saved_row['Updated At']))}"

        if not original and audio_id not in self.missing_in_source:
            self.missing_in_source.append(audio_id)

        self.header.value = self.card_html(
            f"{html.escape(audio_id)}",
            f"<p>{global_index}/{len(self.wav_files)} | Navigator {self.index + 1}/{len(self.session_wavs)} | {reviewed_label}{updated_at}</p>"
        )
        self.render_audio(wav_path)

        original_display = html.escape(original) if original else f"<i>No text found for this audio in {html.escape(str(SOURCE_XLSX))}</i>"
        current_display = html.escape(current) if current else "<i>Empty text</i>"
        self.original_text.value = self.card_html("Original text", original_display)
        self.saved_text.value = self.card_html("Saved text", current_display)
        self.command.value = current
        self.command.disabled = False
        self.next_button.disabled = False
        self.prev_button.disabled = False
        self.quit_button.disabled = False
        self.status.value = self.status_html(self.message)
        self.message = ""

    def save_current(self, final_text=None):
        if not self.session_wavs or self.index < 0 or self.index >= len(self.session_wavs):
            return
        wav_path = self.resolve_wav_path(self.session_wavs[self.index])
        audio_id = wav_path.name
        original = self.source_map.get(audio_id, "")
        if final_text is None:
            final_text = self.command.value

        final_text = str(final_text).strip()
        self.draft_texts[audio_id] = final_text
        decision = "updated" if final_text != original else "kept"
        self.progress_df = self.progress_df[self.progress_df["Audio Id"] != audio_id].copy()
        self.progress_df.loc[len(self.progress_df)] = {
            "Audio Id": audio_id,
            "Original Text": original,
            "Text": final_text,
            "Decision": decision,
            "Updated At": datetime.now().isoformat(timespec="seconds"),
        }
        save_progress(self.progress_df)
        self.refresh_progress_df()
        self.state = load_review_state()
        saved = self.progress_lookup(audio_id, refresh=False)
        if saved is not None:
            self.draft_texts[audio_id] = str(saved["Text"])
        self.message = f"<b>Saved:</b> {html.escape(audio_id)}"

    def save_and_next(self, final_text=None):
        if self.stopped or not self.session_wavs:
            return
        self.save_current(final_text=final_text)
        if self.index >= len(self.session_wavs) - 1:
            self.message = "<b>Saved current text.</b> Already at the last item in the navigator."
        else:
            self.index += 1
        self.render()

    def previous_item(self):
        if self.stopped or not self.session_wavs:
            return
        self.save_current()
        if self.index == 0:
            if self.progress_df.empty:
                self.message = "<b>Warning:</b> No saved review data exists yet, so there are no previous saved items to open."
            else:
                self.message = "<b>Saved current text.</b> Already at the first item in the navigator."
        else:
            self.index -= 1
            prev_audio_id = self.session_wavs[self.index].name
            self.draft_texts[prev_audio_id] = self.current_text_for(prev_audio_id)
            if self.is_reviewed(prev_audio_id):
                self.message = "<b>Saved current text and moved back.</b> Loaded a previously saved item from the progress file."
            else:
                self.message = "<b>Saved current text and moved back.</b>"
        self.render()

    def stop(self):
        self.stopped = True
        refreshed_state = load_review_state()
        self.header.value = self.card_html("Review stopped", f"Progress is saved in <code>{html.escape(str(OUTPUT_XLSX))}</code>.")
        self.audio_output.clear_output()
        self.original_text.value = ""
        self.saved_text.value = ""
        self.command.disabled = True
        self.next_button.disabled = True
        self.prev_button.disabled = True
        self.quit_button.disabled = True
        self.status.value = self.status_html(
            f"<b>Navigator items:</b> {len(self.session_wavs)} | "
            f"<b>Total reviewed:</b> {len(refreshed_state['progress_df'])} | "
            f"<b>Remaining:</b> {len(refreshed_state['pending_wavs'])}"
        )


review_app = WavTextReviewApp()
display(review_app.container)
review_app.render()